In [4]:
# librerias usadas
import sys
from copy import deepcopy
import os
#Abraham Lugo Ramirez
#Elkin Rodrigez
#Cristian Zapata
#
# Revisado y corregido por
# Eduardo Zurek, Ph.D.
# Marzo 4 de 2021, 7:44 p.m.

#######
####
##
#
file = open("resultado.txt", "w") # se abre el archivo de salida

def output(a):
    sys.stdout.write(str(a))

tamaño = 9
tablero = [
    ['','','','','','','','',''],
    ['','','','','','','','',''],
    ['','','','','','','','',''],
    ['','','','','','','','',''],
    ['','','','','','','','',''],
    ['','','','','','','','',''],
    ['','','','','','','','',''],
    ['','','','','','','','',''],
    ['','','','','','','','','']
] #matriz para solucionar sudoku
archivo_entrada = 'matrix3.txt'
#archivo_entrada = 'matrix-hard.txt'
with open(archivo_entrada, 'r') as f: #archivo de entrada con el estado inicial del sudoku
    contenido = f.read()     
    contador = 0
    numero = 0
   
    while(True):
      if(contenido[contador:contador+1] != ' ' and contenido[contador:contador+1] != '\n' and contenido[contador:contador+1] != '\t' and contenido[contador:contador+1] != ','):
        fila = int(numero/9)
        columna = int(numero - (9*fila))            
        if(contenido[contador:contador+1] == '*'):
          tablero[fila][columna] = 0
        else:
          tablero[fila][columna] = int(contenido[contador:contador+1])

        contador = contador+1
        numero = numero + 1
      else:
        contador= contador+1        
      if(tablero[8][8] != ''):
        break
   
                                                                                                        
#función para dibujar la  matriz
def print_field(tablero):
    if not tablero:
        #output('El tablero está vacío')
        file.write('El tablero está vacío')
        return
    for i in range(tamaño):
        for j in range(tamaño):
            celda = tablero[i][j]
            if celda == 0 or isinstance(celda, set):
                #output('*')
                file.write('*')
            else:
                #output(celda)
                file.write(str(celda))
            if (j + 1) % 3 == 0 and j < 8:
                #output(' |')
                file.write(' |')
            if j != 8:
                #output(' ')
                file.write(' ')
        #output('\n')
        file.write(os.linesep)
        if (i + 1) % 3 == 0 and i < 8:
            #output("- - - + - - - + - - -\n")
            file.write("- - - + - - - + - - -\n")
    file.write('________________________ \n')
    #output('________________________\n')

def read(tablero):
   #función para leer casilla respectiva y remplazar el valor 0 por un valor entre 1 y 9
    state = deepcopy(tablero)
    for i in range(tamaño):
        for j in range(tamaño):
            celda = state[i][j]
            if celda == 0:
                state[i][j] = set(range(1,tamaño+1))
    return state

state = read(tablero) #se lee lo que hay en el tablero

#función para  verficar si el sudoku está resuelto
def done(state):
    for fila in state:
        for celda in fila:
            if isinstance(celda, set):
                return False
    return True

#función para expandir un paso
def propagate_step(state):
    new_units = False
    for i in range(tamaño):
        fila = state[i]
        values = set([x for x in fila if not isinstance(x, set)])
        for j in range(tamaño):
            if isinstance(state[i][j], set):
                state[i][j] -= values     
                if len(state[i][j]) == 0:
                    return False, None
    for j in range(tamaño):
        column = [state[x][j] for x in range(tamaño)]
        values = set([x for x in column if not isinstance(x, set)])
        for i in range(tamaño):
            if isinstance(state[i][j], set):
                state[i][j] -= values
                if len(state[i][j]) == 0:
                    return False, None
    for x in range(3):
        for y in range(3):
            values = set()            
            for i in range(3*x, 3*x+3):
                for j in range(3*y, 3*y+3):
                    celda = state[i][j]
                    if not isinstance(celda, set):
                        values.add(celda)
            for i in range(3*x, 3*x+3):
                for j in range(3*y, 3*y+3):
                    if isinstance(state[i][j], set):
                        state[i][j] -= values
                        if len(state[i][j]) == 0:
                            return False, None
    return True, new_units

#funcion para propagar hasta llegar a un punto fijo
def propagate(state):
    while True:
        solvable, new_unit = propagate_step(state)
        if not solvable:
            return False
        if not new_unit:
            return True

# función para contar jugadas
def contar_jugadas(state):
    numero_de_posibilidades = []
    fila = []
    columna = []
    posibilidades = []
    num_total_jugadas = 0
    for i in range(9):
        for j in range(9):
            celda = state[i][j]
            if isinstance(celda, set):
                num_total_jugadas += len(celda)
                numero_de_posibilidades.append(len(celda))
                fila.append(i)
                columna.append(j)
                posibilidades.append(celda)
            else:
                numero_de_posibilidades.append(100)
                fila.append(i)
                columna.append(j)
                posibilidades.append('')
    jugadas = sorted(zip(numero_de_posibilidades,fila,columna,posibilidades))
    return jugadas, num_total_jugadas

#funcion para resolver sudoku
def solve(state):
    solvable = propagate(state)
    if not solvable:
        return None
    if done(state):
        return state
    # Encontrar la variable más restringida en el estado
    jugadas, num_total_jugadas = contar_jugadas(state)
    # se ordenan de menor a mayor
    # las variables más restringidas quedan de primeras
    for num_pos,fil,col,celda in jugadas:
        if isinstance(celda, set):
            # se ordenan los valores priorizando los menos restrictores
            restricciones_generadas = []
            valores = []
            for value in celda:
                new_state = deepcopy(state)
                new_state[fil][col] = value
                new_jugadas, new_num_total_jugadas = contar_jugadas(new_state)
                restricciones_generadas.append(new_num_total_jugadas)
                valores.append(value)
            valores_ordenados = sorted(zip(restricciones_generadas,valores))
            for restric,value in valores_ordenados:
                file.write("Variable más restringida:\n")
                file.write("Fila : ")
                file.write(str(fil))
                file.write(", Columna: ")
                file.write(str(col))
                file.write(", Valores: ")
                file.write(str(celda))
                file.write("\n")
                file.write("Valor que menos restricciones genera: ")
                file.write(str(value))
                file.write("\n")
                file.write("Nuevo estado: ")
                file.write("\n")                
                new_state = deepcopy(state)
                new_state[fil][col] = value
                print_field(new_state)
                file.write("\n") 
                file.write("\n")
                file.write("\n")                                 
                solved = solve(new_state)
                if solved is not None:
                    return solved
            return None
#file.write(str(print_field(solve(state))) + os.linesep)

print_field(solve(state)) #dibujar el sudoku paso a paso 
file.close() #dibujar sudoku en el archivo de salida